# Explainable AI

若有任何問題，歡迎來信至助教信箱： ntu-ml-2021spring-ta@googlegroups.com

## Deadline
- 5/7 release, 5/28 submit

# **Homework 9 - Explainable AI (Part 1 CNN)**

## 掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
import os
# 切換到 Google Drive 裡的工作資料夾
os.chdir('gdrive/My Drive/MLHW_XAI')
!ls

## 環境設定：下載資料集與預訓練模型

In [ ]:
# 下載並解壓縮食物圖片資料集（11 類食物）
!gdown --id '1cYBWwYab3djiaYuOU6CxkYHQyUYws4Ce' --output food.zip
!unzip food.zip

In [ ]:
# 下載預訓練好的 CNN 分類器 checkpoint
!gdown --id '1CShZHsO8oAZwxQkMe7jRtEgSNb2w_OZu' --output checkpoint.pth

In [ ]:
# 安裝 LIME 套件，用於解釋模型預測
!pip install lime==0.1.1.37

## 匯入套件

In [ ]:
import os
import sys
import argparse
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from skimage.segmentation import slic   # 用於 LIME 的圖片分割
from lime import lime_image             # LIME 解釋套件
from pdb import set_trace
from torch.autograd import Variable

## 參數設定

In [ ]:
args = {
    'ckptpath': './checkpoint.pth',    # 模型 checkpoint 路徑
    'dataset_dir': './food/'           # 資料集路徑
}
args = argparse.Namespace(**args)

## 模型定義與載入

In [ ]:
# CNN 分類器架構定義
# 這個模型用來分類 11 種食物，是 HW3 同款架構
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()

        def building_block(indim, outdim):
            # 每個 block 由 Conv2d + BatchNorm + ReLU 組成
            return [
                nn.Conv2d(indim, outdim, 3, 1, 1),
                nn.BatchNorm2d(outdim),
                nn.ReLU(),
            ]

        def stack_blocks(indim, outdim, block_num):
            # 堆疊多個 building_block，最後接 MaxPool 縮小空間維度
            layers = building_block(indim, outdim)
            for i in range(block_num - 1):
                layers += building_block(outdim, outdim)
            layers.append(nn.MaxPool2d(2, 2, 0))
            return layers

        # CNN 特徵提取部分（5 個 stack，逐漸加深 channel 數）
        cnn_list = []
        cnn_list += stack_blocks(3, 128, 3)    # 輸入 RGB 3 channel → 128
        cnn_list += stack_blocks(128, 128, 3)
        cnn_list += stack_blocks(128, 256, 3)
        cnn_list += stack_blocks(256, 512, 1)
        cnn_list += stack_blocks(512, 512, 1)
        self.cnn = nn.Sequential(*cnn_list)

        # 全連接層（分類器部分）
        # 128x128 圖片經過 5 次 MaxPool(2,2) 後空間大小為 4x4
        dnn_list = [
            nn.Linear(512 * 4 * 4, 1024),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(1024, 11),   # 11 個食物類別
        ]
        self.fc = nn.Sequential(*dnn_list)

    def forward(self, x):
        out = self.cnn(x)
        out = out.reshape(out.size()[0], -1)  # 展平 (batch, 512*4*4)
        return self.fc(out)

In [ ]:
# 載入預訓練模型
model = Classifier().cuda()
checkpoint = torch.load(args.ckptpath)
model.load_state_dict(checkpoint['model_state_dict'])
# 應該顯示：<All keys matched successfully>

## 資料集定義與載入

In [ ]:
class FoodDataset(Dataset):
    def __init__(self, paths, labels, mode):
        # mode: 'train' 或 'eval'
        self.paths = paths
        self.labels = labels

        # 訓練模式：加入資料增強（水平翻轉、隨機旋轉）
        trainTransform = transforms.Compose([
            transforms.Resize(size=(128, 128)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
        ])
        # 評估模式：只做 Resize 和轉 Tensor
        evalTransform = transforms.Compose([
            transforms.Resize(size=(128, 128)),
            transforms.ToTensor(),
        ])
        self.transform = trainTransform if mode == 'train' else evalTransform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        X = Image.open(self.paths[index])
        X = self.transform(X)
        Y = self.labels[index]
        return X, Y

    def getbatch(self, indices):
        # 一次取出多張圖片並 stack 成 batch tensor
        images = []
        labels = []
        for index in indices:
            image, label = self.__getitem__(index)
            images.append(image)
            labels.append(label)
        return torch.stack(images), torch.tensor(labels)


def get_paths_labels(path):
    # 依照檔名排序，取得圖片路徑與對應標籤
    def my_key(name):
        return int(name.replace(".jpg", "")) + 1000000 * int(name.split("_")[0])
    imgnames = os.listdir(path)
    imgnames.sort(key=my_key)
    imgpaths = []
    labels = []
    for name in imgnames:
        imgpaths.append(os.path.join(path, name))
        labels.append(int(name.split('_')[0]))
    return imgpaths, labels


train_paths, train_labels = get_paths_labels(args.dataset_dir)
train_set = FoodDataset(train_paths, train_labels, mode='eval')

## 觀察用的 10 張圖片

11 類食物：Bread、Dairy product、Dessert、Egg、Fried food、Meat、Noodles/Pasta、Rice、Seafood、Soup、Vegetable/Fruit

In [ ]:
# 取前 10 張訓練圖片作為觀察對象
img_indices = [i for i in range(10)]
images, labels = train_set.getbatch(img_indices)

fig, axs = plt.subplots(1, len(img_indices), figsize=(15, 8))
for i, img in enumerate(images):
    # img 的 shape 是 (C, H, W)，matplotlib 需要 (H, W, C)，所以用 permute 調換維度
    axs[i].imshow(img.cpu().permute(1, 2, 0))
plt.show()

---
# Question 1 - 4：LIME

LIME（Local Interpretable Model-agnostic Explanations）的核心思想：
- 對原始圖片做局部擾動（遮蔽部分區域），觀察模型輸出如何改變
- 找出哪些區域對預測結果影響最大（正面 or 負面）
- **綠色區域**：對預測有正面貢獻（支持此分類）
- **紅色區域**：對預測有負面貢獻（妨礙此分類）

In [ ]:
def predict(input):
    # LIME 會傳入 numpy array，shape: (batch, H, W, C)
    # 需要轉成 PyTorch tensor，shape: (batch, C, H, W)
    model.eval()
    input = torch.FloatTensor(input).permute(0, 3, 1, 2)
    output = model(input.cuda())
    return output.detach().cpu().numpy()


def segmentation(input):
    # 用 SLIC 演算法把圖片分割成 200 個超像素（superpixel）區塊
    # LIME 會對這些區塊做遮蔽實驗
    return slic(input, n_segments=200, compactness=1, sigma=1)


fig, axs = plt.subplots(1, len(img_indices), figsize=(15, 8))
np.random.seed(16)  # 固定 random seed，讓結果可重現

for idx, (image, label) in enumerate(zip(images.permute(0, 2, 3, 1).numpy(), labels)):
    x = image.astype(np.double)

    # 建立 LIME 解釋器並對此圖片進行解釋
    explainer = lime_image.LimeImageExplainer()
    explaination = explainer.explain_instance(
        image=x,
        classifier_fn=predict,
        segmentation_fn=segmentation
    )

    lime_img, mask = explaination.get_image_and_mask(
        label=label.item(),
        positive_only=False,  # 同時顯示正面和負面影響的區域
        hide_rest=False,      # 不隱藏非重要區域
        num_features=11,      # 最多顯示 11 個重要區塊
        min_weight=0.05       # 只顯示權重 >= 0.05 的區塊
    )
    axs[idx].imshow(lime_img)

plt.show()
plt.close()

---
# Question 5 - 9：Saliency Map

**核心概念：**
- 計算 Loss 對 **輸入圖片像素** 的偏微分（gradient）
- 某個像素的 gradient 絕對值越大，代表該像素對模型判斷的影響越大
- 視覺化成 heatmap，越亮的地方代表越重要

**與一般訓練的差異：**
- 一般訓練：計算 Loss 對**模型參數**的 gradient，更新模型
- Saliency Map：計算 Loss 對**輸入圖片**的 gradient，觀察重要性

In [ ]:
def normalize(image):
    # 將圖片數值正規化到 [0, 1]，方便視覺化
    return (image - image.min()) / (image.max() - image.min())


def compute_saliency_maps(x, y, model):
    model.eval()
    x = x.cuda()

    # 關鍵：讓 PyTorch 追蹤輸入 x 的 gradient
    # 預設 input tensor 不計算 gradient，這裡要手動開啟
    x.requires_grad_()

    # Forward pass
    y_pred = model(x)
    loss_func = torch.nn.CrossEntropyLoss()
    loss = loss_func(y_pred, y.cuda())

    # Backward pass：計算 loss 對 x（圖片像素）的偏微分
    loss.backward()

    # 取每個 pixel 在 3 個 channel（R, G, B）中 gradient 絕對值的最大值
    # dim=1 是 channel 維度，shape: (batch, H, W)
    saliencies, _ = torch.max(x.grad.data.abs().detach().cpu(), dim=1)

    # 對每張圖分別正規化（各自獨立，只看相對大小）
    saliencies = torch.stack([normalize(item) for item in saliencies])
    return saliencies

In [ ]:
saliencies = compute_saliency_maps(images, labels, model)

# 上排：原始圖片；下排：Saliency Map heatmap
fig, axs = plt.subplots(2, len(img_indices), figsize=(15, 8))
for row, target in enumerate([images, saliencies]):
    for column, img in enumerate(target):
        if row == 0:
            # 原始圖片：(C, H, W) → (H, W, C)
            axs[row][column].imshow(img.permute(1, 2, 0).numpy())
        else:
            # Saliency Map：用 hot colormap，越亮代表 gradient 越大
            axs[row][column].imshow(img.numpy(), cmap=plt.cm.hot)

plt.show()
plt.close()

---
# Question 10 - 13：Smooth Grad

**問題：** Saliency Map 的 gradient 常常很 noisy（有很多無意義的高值像素）

**Smooth Grad 解法：**
1. 對輸入圖片加入多次隨機 Gaussian noise
2. 對每個加 noise 的版本分別計算 Saliency Map
3. 把所有 Saliency Map **平均**，noise 相互抵消，真正重要的特徵被保留

參數說明：
- `epoch`：加 noise 的次數（越多越平滑，但越慢）
- `param_sigma_multiplier`：noise 強度控制（相對於圖片的值域）

In [ ]:
def normalize(image):
    return (image - image.min()) / (image.max() - image.min())


def smooth_grad(x, y, model, epoch, param_sigma_multiplier):
    model.eval()

    mean = 0
    # sigma 根據圖片的值域動態調整，讓 noise 強度合理
    sigma = param_sigma_multiplier / (torch.max(x) - torch.min(x)).item()

    # 初始化累加器（用於平均多次 gradient）
    smooth = np.zeros(x.cuda().unsqueeze(0).size())

    for i in range(epoch):
        # 產生 Gaussian noise 並加到圖片上
        noise = Variable(x.data.new(x.size()).normal_(mean, sigma ** 2))
        x_mod = (x + noise).unsqueeze(0).cuda()
        x_mod.requires_grad_()

        # 計算加 noise 圖片的 gradient（同 Saliency Map 做法）
        y_pred = model(x_mod)
        loss_func = torch.nn.CrossEntropyLoss()
        loss = loss_func(y_pred, y.cuda().unsqueeze(0))
        loss.backward()

        # 累加 gradient 絕對值
        smooth += x_mod.grad.abs().detach().cpu().data.numpy()

    # 取平均後正規化
    smooth = normalize(smooth / epoch)
    return smooth


smooth = []
for i, l in zip(images, labels):
    # epoch=500：平均 500 次；param_sigma_multiplier=0.4：noise 強度
    smooth.append(smooth_grad(i, l, model, 500, 0.4))
smooth = np.stack(smooth)
print(smooth.shape)  # (10, 1, 3, 128, 128)

fig, axs = plt.subplots(2, len(img_indices), figsize=(15, 8))
for row, target in enumerate([images, smooth]):
    for column, img in enumerate(target):
        # reshape 成 (3, 128, 128) 再轉成 (H, W, C) 顯示
        axs[row][column].imshow(np.transpose(img.reshape(3, 128, 128), (1, 2, 0)))

plt.show()
plt.close()

---
# Question 14 - 17：Filter Visualization

**目標：** 了解 CNN 某一層的某個 filter 到底在「認什麼」

做兩件事：
1. **Filter Activation**：把真實圖片餵入，看哪些位置 activate 該 filter（該 filter 對什麼地方有反應）
2. **Filter Visualization**：從白噪音出發，用 **Gradient Ascent** 找出最能激活該 filter 的圖片

**技術重點：PyTorch Hook**
- 問題：如何在 forward 過程中「偷看」中間層的輸出？
- 解法：`register_forward_hook` 在某一層 forward 時自動呼叫一個函數，把中間輸出存下來

**Gradient Ascent vs Gradient Descent：**
- 訓練時：Gradient **Descent**，降低 Loss
- Filter Visualization：Gradient **Ascent**，最大化 filter activation（所以 loss 加負號）

In [ ]:
# 查看模型結構，了解各層的 index（cnnid 要對應到 model.cnn 的 index）
model

In [ ]:
def normalize(image):
    return (image - image.min()) / (image.max() - image.min())


layer_activations = None  # 全域變數，用於儲存 hook 捕捉到的中間層輸出


def filter_explanation(x, model, cnnid, filterid, iteration=100, lr=1):
    """
    x: 輸入圖片 batch
    cnnid: 要觀察的 CNN 層 index（對應 model.cnn[cnnid]）
    filterid: 要觀察的 filter index
    iteration: Gradient Ascent 的迭代次數
    lr: Gradient Ascent 的 learning rate
    """
    model.eval()

    def hook(model, input, output):
        # 這個函數在 model.cnn[cnnid] forward 完後自動被呼叫
        # output 就是該層的輸出（activation map）
        global layer_activations
        layer_activations = output

    # 在指定層註冊 hook
    hook_handle = model.cnn[cnnid].register_forward_hook(hook)

    # ===== Part 1: Filter Activation（真實圖片） =====
    model(x.cuda())  # forward，hook 會自動把中間層輸出存到 layer_activations

    # 取出指定 filter 的 activation map
    # layer_activations shape: (batch, num_filters, H, W)
    filter_activations = layer_activations[:, filterid, :, :].detach().cpu()

    # ===== Part 2: Filter Visualization（Gradient Ascent） =====
    x = x.cuda()
    x.requires_grad_()  # 讓輸入圖片可以被更新
    optimizer = Adam([x], lr=lr)  # 優化目標是輸入圖片 x

    for iter in range(iteration):
        optimizer.zero_grad()
        model(x)  # forward，順便觸發 hook 更新 layer_activations

        # 目標：最大化指定 filter 的 activation 總和
        # 加負號是因為 optimizer 做的是最小化，所以 -activation = 最大化 activation
        objective = -layer_activations[:, filterid, :, :].sum()
        objective.backward()  # 計算對輸入圖片的 gradient
        optimizer.step()      # 更新輸入圖片，讓 filter 更容易被激活

    filter_visualizations = x.detach().cpu().squeeze()

    # 記得移除 hook，否則後續的 forward 都會觸發它
    hook_handle.remove()

    return filter_activations, filter_visualizations

In [ ]:
# 觀察較淺層（cnnid=6）的 filter 0
# 淺層 filter 通常偵測低階特徵（邊緣、顏色等）
images, labels = train_set.getbatch(img_indices)
filter_activations, filter_visualizations = filter_explanation(
    images, model, cnnid=6, filterid=0, iteration=100, lr=0.1
)

fig, axs = plt.subplots(3, len(img_indices), figsize=(15, 8))
for i, img in enumerate(images):
    axs[0][i].imshow(img.permute(1, 2, 0))           # 第一排：原始圖片
for i, img in enumerate(filter_activations):
    axs[1][i].imshow(normalize(img))                  # 第二排：Filter Activation heatmap
for i, img in enumerate(filter_visualizations):
    axs[2][i].imshow(normalize(img.permute(1, 2, 0))) # 第三排：Filter Visualization（Gradient Ascent 結果）
plt.show()
plt.close()

In [ ]:
# 觀察較深層（cnnid=23）的 filter 0
# 深層 filter 通常偵測高階特徵（紋理、物件部件等）
images, labels = train_set.getbatch(img_indices)
filter_activations, filter_visualizations = filter_explanation(
    images, model, cnnid=23, filterid=0, iteration=100, lr=0.1
)

fig, axs = plt.subplots(3, len(img_indices), figsize=(15, 8))
for i, img in enumerate(images):
    axs[0][i].imshow(img.permute(1, 2, 0))
for i, img in enumerate(filter_activations):
    axs[1][i].imshow(normalize(img))
for i, img in enumerate(filter_visualizations):
    axs[2][i].imshow(normalize(img.permute(1, 2, 0)))
plt.show()
plt.close()

---
# Question 18 - 20：Integrated Gradients

**Saliency Map 的問題：** Gradient Saturation
- 若模型對某個特徵已非常確定，gradient 可能趨近 0（飽和），即使該特徵很重要

**Integrated Gradients 解法：**
- 設定一個 **baseline**（全黑圖 = 零輸入）
- 在 baseline → 原圖 的路徑上，均勻取多個中間點（`x̄ + α(x - x̄)`，α 從 0 到 1）
- 對每個中間點計算 gradient，然後**積分**（數值近似：取平均）
- 公式：`(x_i - x̄_i) · ∫ ∂S_c(x̃) / ∂(x̃_i) dα`

**直覺：** 整合從無到有的整個過程，避免只看終點 gradient 帶來的飽和問題

In [ ]:
class IntegratedGradients():
    def __init__(self, model):
        self.model = model
        self.gradients = None
        self.model.eval()

    def generate_images_on_linear_path(self, input_image, steps):
        # 在 baseline（全零）到 input_image 的線性路徑上，產生 steps 個中間圖片
        # α = 0/steps, 1/steps, ..., (steps-1)/steps
        xbar_list = [input_image * step / steps for step in range(steps)]
        return xbar_list

    def generate_gradients(self, input_image, target_class):
        # 計算模型輸出對輸入圖片的 gradient（針對目標類別）
        input_image.requires_grad = True
        model_output = self.model(input_image)
        self.model.zero_grad()

        # 建立 one-hot 向量，讓 backward 只針對目標類別計算 gradient
        one_hot_output = torch.FloatTensor(1, model_output.size()[-1]).zero_().cuda()
        one_hot_output[0][target_class] = 1
        model_output.backward(gradient=one_hot_output)

        self.gradients = input_image.grad
        # 轉成 numpy，去掉 batch 維度 [0]
        gradients_as_arr = self.gradients.data.cpu().numpy()[0]
        return gradients_as_arr

    def generate_integrated_gradients(self, input_image, target_class, steps):
        # 在線性路徑上的每個點計算 gradient，然後平均（數值積分）
        xbar_list = self.generate_images_on_linear_path(input_image, steps)
        integrated_grads = np.zeros(input_image.size())

        for xbar_image in xbar_list:
            single_integrated_grad = self.generate_gradients(xbar_image, target_class)
            integrated_grads = integrated_grads + single_integrated_grad / steps

        # 去掉 batch 維度，回傳 (3, H, W) 的 numpy array
        return integrated_grads[0]


def normalize(image):
    return (image - image.min()) / (image.max() - image.min())

In [ ]:
# 把圖片搬到 GPU
images, labels = train_set.getbatch(img_indices)
images = images.cuda()

In [ ]:
IG = IntegratedGradients(model)
integrated_grads = []

for i, img in enumerate(images):
    img = img.unsqueeze(0)  # 加 batch 維度：(3, H, W) → (1, 3, H, W)
    # steps=10：在路徑上取 10 個中間點做數值積分
    integrated_grads.append(IG.generate_integrated_gradients(img, labels[i], 10))

fig, axs = plt.subplots(2, len(img_indices), figsize=(15, 8))
for i, img in enumerate(images):
    axs[0][i].imshow(img.cpu().permute(1, 2, 0))         # 第一排：原始圖片
for i, img in enumerate(integrated_grads):
    # np.moveaxis(img, 0, -1) 把 (3, H, W) → (H, W, 3)
    axs[1][i].imshow(np.moveaxis(normalize(img), 0, -1)) # 第二排：Integrated Gradients 結果
plt.show()
plt.close()

---
# **Homework 9 - Explainable AI (Part 2 BERT)**

# Question 21 - 24：Attention Visualization

直接在 exBERT 網站上操作觀察：https://exbert.net/exBERT.html

觀察目標：
1. 不同 attention head 各自負責什麼功能（語法關係、指代、位置等）
2. 模型如何預測被 [MASK] 遮住的詞

In [ ]:
from IPython import display
# 在 Colab 中嵌入 exBERT 網站
display.IFrame("https://exbert.net/exBERT.html", width=1600, height=1600)

# 匯入 Q25-30 所需套件

In [ ]:
# 安裝 Hugging Face transformers（用於載入 BERT 模型和 tokenizer）
!pip install transformers==4.5.0

import numpy as np
import random
import torch

from sklearn.decomposition import PCA          # 主成分分析，用於降維視覺化
from sklearn.metrics import pairwise_distances  # 計算 pairwise 距離矩陣
from transformers import BertModel, BertTokenizerFast

# 下載台北思源黑體，讓 matplotlib 能顯示繁體中文
!gdown --id '1WOjwyN_wimGfrw0kE2nvp6OxFimch3H5' --output taipei_sans_tc_beta.ttf

from matplotlib.font_manager import FontProperties
import matplotlib.pyplot as plt

myfont = FontProperties(fname=r'taipei_sans_tc_beta.ttf')
plt.rcParams['figure.figsize'] = [12, 10]


def same_seeds(seed):
    # 固定所有隨機種子，確保結果可重現
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


same_seeds(0)

---
# Question 25 - 27：Embedding Visualization

**任務情境：** Tom 有 3 個 BERT 模型（只有其中 1 個有 fine-tune 過 QA 任務），請透過觀察 embedding 來判斷哪個模型是 QA 模型。

**方法：** 用 PCA 將 BERT 的 768 維 hidden state 降至 2 維，畫出每個 token 的位置
- 藍色菱形：答案詞
- 紅色：問題中的詞
- 綠色：文章中的詞

**觀察重點：** fine-tune 過 QA 的模型，在某些層應該會把「答案詞」cluster 到「問題相關詞」附近

In [ ]:
# 下載 3 個模型的 hidden states 資料
!gdown --id '17AZ2KAUFpYHJOC0DHet3cQWYE5sX2yxo' --output hw9_bert.zip
!unzip hw9_bert.zip

In [ ]:
# 載入 3 個模型各自的 tokenizer
tokenizer1 = BertTokenizerFast.from_pretrained("hw9_bert/tokenizer1")
tokenizer2 = BertTokenizerFast.from_pretrained("hw9_bert/tokenizer2")
tokenizer3 = BertTokenizerFast.from_pretrained("hw9_bert/tokenizer3")
tokenizers = [tokenizer1, tokenizer2, tokenizer3]

In [ ]:
# 三個 QA 問題的文章、問題、答案
contexts, questions, answers = [], [], []

# Question 1
contexts += ['Currently detention is one of the most common punishments in schools in the United States, the UK, Ireland, Singapore and other countries. \
            It requires the pupil to remain in school at a given time in the school day (such as lunch, recess or after school); or even to attend \
            school on a non-school day, e.g. "Saturday detention" held at some schools. During detention, students normally have to sit in a classroom \
            and do work, write lines or a punishment essay, or sit quietly.']
questions += ['What is a common punishment in the UK and Ireland?']
answers += ['detention']

# Question 2
contexts += ['Wolves are afraid of cats. Sheep are afraid of wolves. Mice are afraid of sheep. Gertrude is a mouse. Jessica is a mouse. \
            Emily is a wolf. Cats are afraid of sheep. Winona is a wolf.']
questions += ['What is Emily afraid of?']
answers += ['cats']

# Question 3
contexts += ["Nikola Tesla (Serbian Cyrillic: Никола Тесла; 10 July 1856 – 7 January 1943) was a Serbian American inventor, electrical engineer, \
            mechanical engineer, physicist, and futurist best known for his contributions to the design of the modern alternating current \
            (AC) electricity supply system."]
questions += ["In what year was Nikola Tesla born?"]
answers += ["1856"]

In [ ]:
# ===== TODO：改變這裡的設定來觀察不同模型和問題 =====
MODEL = 1    # 選擇模型：1, 2, 3
QUESTION = 1 # 選擇問題：1, 2, 3

In [ ]:
# 將問題和文章 tokenize，取得 input_ids
inputs = tokenizers[MODEL-1](questions[QUESTION-1], contexts[QUESTION-1], return_tensors='pt')

# 找出問題和文章在 token 序列中的位置範圍
# BERT 格式：[CLS] 問題 [SEP] 文章 [SEP]
# 102 是 [SEP] 的 token id
question_start, question_end = 1, inputs['input_ids'][0].tolist().index(102) - 1
context_start, context_end = question_end + 2, len(inputs['input_ids'][0]) - 2

# 載入預先計算好的 hidden states（避免每次都要跑完整模型）
outputs_hidden_states = torch.load(f"hw9_bert/output/model{MODEL}_q{QUESTION}")

# 遍歷 BERT 12 層的 hidden states（第 0 個是 embedding 層，跳過）
for layer_index, embeddings in enumerate(outputs_hidden_states[1:]):
    # embeddings shape: (1, sequence_length, 768)
    # 用 PCA 降維從 768 → 2 維，方便畫圖
    reduced_embeddings = PCA(n_components=2, random_state=0).fit_transform(embeddings[0])

    for i, token_id in enumerate(inputs['input_ids'][0]):
        x, y = reduced_embeddings[i]  # 2D 座標
        word = tokenizers[MODEL-1].decode(token_id)  # token id 轉回文字

        # 依據 token 所在位置（答案/問題/文章）用不同顏色標記
        if word in answers[QUESTION-1].split():
            plt.scatter(x, y, color='blue', marker='d')  # 答案詞：藍色菱形
        elif question_start <= i <= question_end:
            plt.scatter(x, y, color='red')               # 問題詞：紅色
        elif context_start <= i <= context_end:
            plt.scatter(x, y, color='green')             # 文章詞：綠色
        else:
            continue  # 跳過特殊 token [CLS], [SEP]
        plt.text(x + 0.1, y + 0.2, word, fontsize=12)

    # 圖例和標題
    plt.plot([], label='answer', color='blue', marker='d')
    plt.plot([], label='question', color='red', marker='o')
    plt.plot([], label='context', color='green', marker='o')
    plt.legend(loc='best')
    plt.title('Layer ' + str(layer_index + 1))
    plt.show()

---
# Question 28 - 30：Embedding Analysis

**任務：** 比較「蘋果」這個詞在不同句子中的 BERT embedding 相似度
- 前幾句的「蘋果」指水果
- 後幾句的「蘋果」指 Apple 公司

**觀察目標：**
1. BERT 能否區分同一個詞在不同語境下的不同含義（contextualized embedding）？
2. 不同層的 embedding 對語義的捕捉有何差異？

**兩種比較方式：**
- **Euclidean Distance（L2 距離）**：距離越小代表越相似
- **Cosine Similarity（餘弦相似度）**：值越大（越接近 1）代表越相似，與向量長度無關

In [ ]:
# 載入中文 BERT 模型（bert-base-chinese）
# output_hidden_states=True：讓模型回傳所有層的 hidden states
model = BertModel.from_pretrained('bert-base-chinese', output_hidden_states=True).eval()
tokenizer = BertTokenizerFast.from_pretrained('bert-base-chinese')

In [ ]:
# 10 個含有「蘋果」的句子：前半指水果，後半指 Apple 公司
sentences = []
sentences += ["今天買了蘋果來吃"]          # 水果
sentences += ["進口蘋果（富士)平均每公斤下跌12.3%"]  # 水果
sentences += ["蘋果茶真難喝"]              # 水果
sentences += ["老饕都知道智利的蘋果季節即將到來"]    # 水果
sentences += ["進口蘋果因防止水分流失故添加人工果糖"] # 水果
sentences += ["蘋果即將於下月發振新款iPhone"]       # Apple 公司
sentences += ["蘋果獲新Face ID專利"]              # Apple 公司
sentences += ["今天買了蘋果手機"]                  # Apple 公司
sentences += ["蘋果的股價又跌了"]                  # Apple 公司
sentences += ["蘋果押寶指紋辨識技術"]              # Apple 公司

# 每個句子中「蘋果」第一個字的字元位置（用於找到對應的 token）
select_word_index = [4, 2, 0, 8, 2, 0, 0, 4, 0, 0]

In [ ]:
# ===== TODO：以下是需要實作和調整的部分 =====

def euclidean_distance(a, b):
    # Euclidean distance（L2 norm）：值越小代表越相似
    return np.linalg.norm(a - b)


def cosine_similarity(a, b):
    # Cosine Similarity：計算兩個向量夾角的餘弦值
    # 公式：cos(θ) = (a · b) / (||a|| * ||b||)
    # 值域 [-1, 1]，越接近 1 代表越相似
    # 注意：pairwise_distances 計算的是「距離」（越小越相似）
    # 所以用 1 - cosine_similarity，讓相似的變成「距離小」
    dot_product = np.dot(a, b)                       # 內積
    norm_a = np.linalg.norm(a)                       # a 的 L2 norm
    norm_b = np.linalg.norm(b)                       # b 的 L2 norm
    similarity = dot_product / (norm_a * norm_b)     # 餘弦相似度
    return 1 - similarity                            # 轉成距離（相似度越高，距離越小）


# 選擇比較指標：euclidean_distance 或 cosine_similarity
METRIC = cosine_similarity


def get_select_embedding(output, tokenized_sentence, select_word_index):
    # 選擇要觀察的層（0 = embedding 層，1-12 = BERT 12 層 attention 輸出）
    LAYER = 12  # 可以改成 0~12 觀察不同層的差異

    # 取得指定層的 hidden state，shape: (sequence_length, 768)
    hidden_state = output.hidden_states[LAYER][0]

    # 將字元位置轉換成 tokenizer 的 token 位置
    # 中文 BERT 每個字通常對應一個 token，但有時一個詞會被切成多個 token
    select_token_index = tokenized_sentence.word_to_tokens(select_word_index).start

    # 回傳該 token 的 embedding（轉成 numpy）
    return hidden_state[select_token_index].numpy()

In [ ]:
# 對 10 個句子進行 tokenize 和 encode
tokenized_sentences = [tokenizer(sentence, return_tensors='pt') for sentence in sentences]

# 用 torch.no_grad() 關閉 gradient 計算（推論階段不需要 gradient，節省記憶體）
with torch.no_grad():
    outputs = [model(**tokenized_sentence) for tokenized_sentence in tokenized_sentences]

# 取出每個句子中「蘋果」的 embedding，shape: (10, 768)
embeddings = [
    get_select_embedding(outputs[i], tokenized_sentences[i], select_word_index[i])
    for i in range(len(outputs))
]

# 計算 pairwise 相似度矩陣，shape: (10, 10)
# metric 可以是 euclidean_distance 或 cosine_similarity（已轉成距離）
similarity_matrix = pairwise_distances(embeddings, metric=METRIC)

# ===== 畫出相似度矩陣 =====
plt.rcParams['figure.figsize'] = [12, 10]
plt.imshow(similarity_matrix)  # 用顏色深淺顯示距離大小
plt.colorbar()                 # 顏色條
plt.yticks(
    ticks=range(len(sentences)),
    labels=sentences,
    fontproperties=myfont  # 使用下載的中文字型
)
plt.title('Cosine Similarities of BERT Embeddings')

# 在每個格子中顯示數值
for (i, j), label in np.ndenumerate(similarity_matrix):
    plt.text(i, j, '{:.2f}'.format(label), ha='center', va='center')

plt.show()